# Data Ingest
### Purpose
Exploratory notebook testing data ingest processes. The notebook extracts DHRs from the Home Office library and store as PDF in ```/data/raw/```

In [ ]:
# Get the html from the Home Office website and parse it with BeautifulSoup
import requests
from bs4 import BeautifulSoup

# dummy url for testing
url = 'https://homicide-review.homeoffice.gov.uk/?pagination[pageNumber]=0'

# fetch the page
r = requests.get(url, timeout=3)

# check the status code
r.raise_for_status()

# extract the html
html = r.text

# parse the html
soup = BeautifulSoup(html, "html.parser")

In [163]:
# CSS selectors
results_selector = "section[aria-label^='DHR result']"
subsection_selector = ".govuk-grid-row.govuk-\!-padding-top-2 .govuk-grid-column-one-quarter .govuk-label.dhrr-results-card--detail-value"
title_selector = "h3.govuk-heading-s"
download_selector = ".govuk-grid-row.govuk-\!-padding-top-2 .govuk-grid-column-full .govuk-button-group.govuk-\!-margin-0 a[href^='/download/']"

# <div class="govuk-grid-column-full">
                
#       <div class="govuk-button-group govuk-!-margin-0">
#           <a href="/download/699496a0c7f5132b09d6c679" aria-label="View Susan - Lincoln - 2018/a - DHR in a new tab" class="govuk-button govuk-button--secondary govuk-!-margin-bottom-1" data-module="govuk-button" target="_blank">
#               View
#           </a>
#           <a href="/download/699496a0c7f5132b09d6c679" download="Susan - Lincoln - 2018/a - DHR" aria-label="Download Susan - Lincoln - 2018/a - DHR" class="govuk-button govuk-button--secondary govuk-!-margin-bottom-1" data-module="govuk-button">
#               Download
#           </a>
          
#     </div>

#                 <hr class="govuk-section-break--visible">
#             </div>

In [164]:
# Identify each DHR section on page
dhr_results = soup.select(results_selector)
print(f"Found {len(dhr_results)} DHR sections.")

Found 50 DHR sections.


In [165]:
# Loop through each DHR result
import pandas as pd

df = pd.DataFrame(columns=["title", "csp", "region", "upload_date", "death_date", "download_id"])

for section in dhr_results:
    print_output="verbose"
    
    # Empty list to hold data extracted from HTML
    extracted_data = []

    # Extract title
    title_element = section.select_one(title_selector)
    title = title_element.get_text(strip=True) if title_element else "N/A"

    # Extract CSP, region, date of upload, date of death
    dhr_subsections = section.select(subsection_selector)
    csp = dhr_subsections[0].get_text().strip() if dhr_subsections else "N/A"

    csp_region = dhr_subsections[1].get_text().strip() if len(dhr_subsections) > 1 else "N/A"

    upload_date = dhr_subsections[2].get_text().strip() if len(dhr_subsections) > 2 else "N/A"

    death_date = dhr_subsections[3].get_text().strip() if len(dhr_subsections) > 3 else "N/A"

    # Extract the download ID
    download_tag = section.select_one(download_selector)['href'] if section.select_one(download_selector) else None
    download_id = download_tag.split('/')[-1] if download_tag else "N/A"

    # Append extracted data to list
    extracted_data.append({
        "title": title,
        "csp": csp,
        "region": csp_region,
        "upload_date": upload_date,
        "death_date": death_date,
        "download_id": download_id
    })

    # Auditing
    if print_output == 'verbose':
        print(f"Title: {title}")
        print(f"Community Service Partnership: {csp}")
        print(f"Region: {csp_region}")
        print(f"Upload Date: {upload_date}")
        print(f"Death Date: {death_date}")
        print(f"Download ID: {download_id}")
        print("-------end of section-------")
    elif print_output == 'title_only':
        print(f"Title: {title}")
    else:
        pass

    # Add extracted data to dataframe
    df = pd.concat([df, pd.DataFrame(extracted_data)], ignore_index=True)


# Add extracted data to dataframe
df.shape

Title: Susan
Community Service Partnership: Lincoln
Region: East Midlands
Upload Date: February 2026
Death Date: September 2018
Download ID: 699496a0c7f5132b09d6c679
-------end of section-------
Title: Bobo
Community Service Partnership: Norfolk
Region: East
Upload Date: February 2026
Death Date: March 2020
Download ID: 698df6801741be0d9e0d749d
-------end of section-------
Title: Irene
Community Service Partnership: Norfolk
Region: East
Upload Date: February 2026
Death Date: April 2022
Download ID: 698de8f6c22a2d68aadd9177
-------end of section-------
Title: John
Community Service Partnership: Ealing
Region: Greater London
Upload Date: February 2026
Death Date: July 2020
Download ID: 698ddd331741be0d9e0d749c
-------end of section-------
Title: Rose
Community Service Partnership: Manchester
Region: North West
Upload Date: February 2026
Death Date: July 2021
Download ID: 698cafa3fea86e62b5070e44
-------end of section-------
Title: Elaine
Community Service Partnership: Southend-on-Sea
Reg

(50, 6)

In [188]:
# Pagination testing
from bs4 import BeautifulSoup
with open('/Users/darren.cook/Documents/Code/dhr-information-extraction/data/raw/html/home_office_library.html', 'r', encoding='utf-8') as f:
      soup = BeautifulSoup(f, "html.parser")


BASE = "https://homicide-review.homeoffice.gov.uk/"
PAGE_SIZE = 100


# Extracting the 'full results' section from the html
showing_results_selector = ".govuk-grid-column-two-thirds .govuk-grid-row .govuk-grid-column-full .govuk-caption-m"

total_results_caption = soup.select(showing_results_selector)
for i, element in enumerate(total_results_caption):
    if total_results_caption[i].get_text(strip=True).startswith("Showing results"):
         total_results = total_results_caption[i].get_text(strip=True)
         break
print(int(total_results.split()[-1]))

#
    

#     a[href^='/download/']

647


In [198]:
def fetch_html_content(url, timeout: int = 3, page_count: int = 7, save_html: bool = True):
    for page in range(page_count):
        page_url = f"{url}?pagination%5BpageNumber%5D={page}&pagination%5BpageSize%5D=100"
        print(page_url)

fetch_html_content(url='https://homicide-review.homeoffice.gov.uk/')

https://homicide-review.homeoffice.gov.uk/?pagination%5BpageNumber%5D=0&pagination%5BpageSize%5D=100
https://homicide-review.homeoffice.gov.uk/?pagination%5BpageNumber%5D=1&pagination%5BpageSize%5D=100
https://homicide-review.homeoffice.gov.uk/?pagination%5BpageNumber%5D=2&pagination%5BpageSize%5D=100
https://homicide-review.homeoffice.gov.uk/?pagination%5BpageNumber%5D=3&pagination%5BpageSize%5D=100
https://homicide-review.homeoffice.gov.uk/?pagination%5BpageNumber%5D=4&pagination%5BpageSize%5D=100
https://homicide-review.homeoffice.gov.uk/?pagination%5BpageNumber%5D=5&pagination%5BpageSize%5D=100
https://homicide-review.homeoffice.gov.uk/?pagination%5BpageNumber%5D=6&pagination%5BpageSize%5D=100
